In [ ]:
# ==============================================
# 01_baseline_text_classification_final.ipynb
# Phase 1 – Machine Learning for Vishing Detection (FYP)
# ==============================================

import os

# --- Windows + TensorFlow logging safety ---
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"     # silence TF C++ logs
os.environ["PYTHONUTF8"] = "1"               # force UTF-8 encoding
os.environ["PYTHONIOENCODING"] = "utf-8"



# --- Standard imports ---
import unicodedata
import tensorflow as tf
import re
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter

import matplotlib.pyplot as plt

from sklearn.model_selection import GroupShuffleSplit
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
    balanced_accuracy_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

import joblib

# --- Deep learning ---
from tensorflow.keras import layers, Model


print("✅ Imports loaded safely.")


In [ ]:
# === 2) LOAD DATASET ===


BASE_DIR = Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()
data_path = BASE_DIR / "data" / "english_dataset_final_v2.csv"

df = pd.read_csv(data_path)
df["transcript"] = df["transcript"].astype(str)
df["label"] = df["label"].astype(str).str.lower().str.strip()

print(f"✅ Loaded: {data_path}")
print("Rows:", len(df))
print("Label counts:\n", df["label"].value_counts())

# === 3) ENGLISH-SAFE NORMALIZATION (fixes .keras UnicodeEncodeError on Windows) ===
def normalize_english(s: str) -> str:
    s = str(s)

    # remove invisible junk
    s = s.replace("\u200b", " ")

    # normalize unicode forms
    s = unicodedata.normalize("NFKC", s)

    # normalize smart punctuation to plain ASCII equivalents
    s = (s.replace("’", "'").replace("‘", "'")
           .replace("“", '"').replace("”", '"')
           .replace("–", "-").replace("—", "-"))

    # IMPORTANT: strip non-ascii characters (prevents .keras save crashing on Windows cp1252)
    s = s.encode("ascii", "ignore").decode("ascii")

    # tidy spaces + lowercase
    s = re.sub(r"\s+", " ", s).strip().lower()
    return s

# quick visibility: how many rows had non-ascii before cleaning
non_ascii_before = df["transcript"].apply(lambda x: any(ord(c) > 127 for c in str(x))).sum()
print("Non-ASCII rows before cleaning:", non_ascii_before)

df["clean_text"] = df["transcript"].apply(normalize_english)

# drop empties created by stripping non-ascii
df = df[df["clean_text"].str.len() > 0].reset_index(drop=True)

non_ascii_after = df["clean_text"].apply(lambda x: any(ord(c) > 127 for c in str(x))).sum()
print("Non-ASCII rows after cleaning:", non_ascii_after)

# === 4) CHECK DUPLICATES ===
dup_count = df["clean_text"].duplicated().sum()
print("Duplicate transcripts:", dup_count)

# drop exact duplicates (recommended)
df = df.drop_duplicates(subset=["clean_text", "label"]).reset_index(drop=True)
print("After dropping exact duplicates:", len(df))
print("Label counts after drop:\n", df["label"].value_counts())

# === 5) LEAKAGE-SAFE SPLIT using groups ===
groups = pd.factorize(df["clean_text"])[0]

gss = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
train_idx, test_idx = next(gss.split(df["clean_text"], df["label"], groups=groups))

train_df = df.iloc[train_idx].reset_index(drop=True)
test_df  = df.iloc[test_idx].reset_index(drop=True)

X_train = train_df["clean_text"].values
y_train = train_df["label"].values
X_test  = test_df["clean_text"].values
y_test  = test_df["label"].values

print("\n✅ Split done (leakage-safe).")
print("Train:", Counter(y_train))
print("Test :", Counter(y_test))

overlap = set(train_df["clean_text"]).intersection(set(test_df["clean_text"]))
print("Identical transcript overlap across splits:", len(overlap))


In [ ]:
# === 6) FEATURE EXTRACTOR (Korean-friendly char n-grams) ===
tfidf_char = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=2,
    max_df=0.95,
    max_features=30000
)

# === 7) MODELS ===
lr = Pipeline([
    ("tfidf", tfidf_char),
    ("clf", LogisticRegression(max_iter=3000, class_weight="balanced"))
])

rf = Pipeline([
    ("tfidf", tfidf_char),
    ("clf", RandomForestClassifier(
        n_estimators=400,
        random_state=42,
        class_weight="balanced_subsample",
        n_jobs=-1
    ))
])

# For confidence, calibrate SVM (gives predict_proba)
svm_base = Pipeline([
    ("tfidf", tfidf_char),
    ("clf", LinearSVC(class_weight="balanced", random_state=42))
])
svm = CalibratedClassifierCV(svm_base, method="sigmoid", cv=3)

models = {
    "SVM (Calibrated)": svm,
    "Logistic Regression": lr,
    "Random Forest": rf
}

results = {}

print("\n=== BASELINE RESULTS (Leakage-safe split) ===")
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    acc = accuracy_score(y_test, pred)
    bacc = balanced_accuracy_score(y_test, pred)
    f1m = f1_score(y_test, pred, average="macro")

    print(f"\n--- {name} ---")
    print(classification_report(y_test, pred, digits=3))
    print("Accuracy:", round(acc, 4),
          "| Balanced Acc:", round(bacc, 4),
          "| F1-macro:", round(f1m, 4))

    results[name] = {"acc": acc, "bacc": bacc, "f1_macro": f1m}

print("\n✅ Baseline training complete.")


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

best_name = max(results, key=lambda k: results[k]["f1_macro"])
best_model = models[best_name]

print("✅ Best baseline by F1-macro:", best_name)

pred_best = best_model.predict(X_test)
cm = confusion_matrix(y_test, pred_best, labels=["safe", "vishing"])

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["safe", "vishing"])
disp.plot()
plt.title(f"Confusion Matrix — {best_name}")
plt.show()

# How often does it predict each class?
print("Predicted label distribution:", Counter(pred_best))


In [ ]:
# === 8) KERAS MODEL (TextVectorization inside model; FIXED dtype) ===
import tensorflow as tf
import numpy as np
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping

# Force tf.string tensors (avoids "Invalid dtype: strXXXXX")
X_train_tf = tf.constant(list(X_train), dtype=tf.string)
X_test_tf  = tf.constant(list(X_test), dtype=tf.string)

# Shape to (batch, 1) because our Input is shape=(1,)
X_train_tf = tf.reshape(X_train_tf, (-1, 1))
X_test_tf  = tf.reshape(X_test_tf, (-1, 1))

# Binary target: vishing=1, safe=0
y_train_bin = (y_train == "vishing").astype(np.int32)
y_test_bin  = (y_test == "vishing").astype(np.int32)

# Text vectorizer (character-level)
max_tokens = 2000
seq_len = 300

text_vec = layers.TextVectorization(
    standardize=None,
    split="character",
    max_tokens=max_tokens,
    output_mode="int",
    output_sequence_length=seq_len
)

# Adapt on training only
text_vec.adapt(X_train_tf)

# Model expects (batch, 1) of strings
inputs = layers.Input(shape=(1,), dtype=tf.string)
x = text_vec(inputs)

x = layers.Embedding(input_dim=max_tokens, output_dim=64)(x)
x = layers.Conv1D(128, 5, activation="relu")(x)
x = layers.GlobalMaxPooling1D()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(64, activation="relu")(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

nn_model = Model(inputs, outputs)
nn_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# Class weights
count_safe = int(np.sum(y_train_bin == 0))
count_vish = int(np.sum(y_train_bin == 1))
total = int(len(y_train_bin))
class_weight = {
    0: total / (2 * count_safe),
    1: total / (2 * count_vish)
}
print("Class_weight:", class_weight)

history = nn_model.fit(
    X_train_tf, y_train_bin,
    validation_data=(X_test_tf, y_test_bin),
    epochs=20,
    batch_size=32,
    class_weight=class_weight,
    verbose=1,
    callbacks=[EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)]
)

# Evaluate
nn_probs = nn_model.predict(X_test_tf, verbose=0).reshape(-1)
nn_pred = (nn_probs >= 0.5).astype(int)

nn_acc = accuracy_score(y_test_bin, nn_pred)
nn_bacc = balanced_accuracy_score(y_test_bin, nn_pred)
nn_f1m = f1_score(y_test_bin, nn_pred, average="macro")

print("\n=== NEURAL NET RESULTS ===")
print("Accuracy:", round(nn_acc, 4),
      "| Balanced Acc:", round(nn_bacc, 4),
      "| F1-macro:", round(nn_f1m, 4))


In [ ]:
# 9) SAVE MODELS FOR PROTOTYPE (Windows-stable)

save_dir = BASE_DIR / "models"
save_dir.mkdir(exist_ok=True)

# Save classical ML models
joblib.dump(models["SVM (Calibrated)"], save_dir / "svm_model.pkl")
joblib.dump(models["Logistic Regression"], save_dir / "logistic_regression_model.pkl")
joblib.dump(models["Random Forest"], save_dir / "rf_model.pkl")

# Save neural network in H5 format (Windows-safe)
nn_model.save(str(save_dir / "neural_network.keras"))

print("Models saved successfully:")
print(" - svm_model.pkl")
print(" - logistic_regression_model.pkl")
print(" - rf_model.pkl")
print(" - neural_network.keras")

In [ ]:
# === 10) PROTOTYPE SYNC TEST (FIXED) ===
import tensorflow as tf
import numpy as np
import joblib
from tensorflow.keras.models import load_model

svm_loaded = joblib.load(save_dir / "svm_model.pkl")
lr_loaded  = joblib.load(save_dir / "logistic_regression_model.pkl")
rf_loaded  = joblib.load(save_dir / "rf_model.pkl")
nn_loaded  = load_model(save_dir / "neural_network.keras")  # or .h5 if you saved that

samples = [
    "Hello, this is a reminder that your appointment is scheduled for tomorrow at 10 AM.",
    "This is the bank security department. Your account has been suspended due to suspicious activity."
]

print("\n=== PROTOTYPE SYNC TEST ===")
print("SVM:", svm_loaded.predict(samples))
print("LR :", lr_loaded.predict(samples))
print("RF :", rf_loaded.predict(samples))

# NN needs tf.string with shape (batch, 1)
samples_tf = tf.constant(samples, dtype=tf.string)
samples_tf = tf.reshape(samples_tf, (-1, 1))

nn_probs = nn_loaded.predict(samples_tf, verbose=0).reshape(-1)
nn_labels = np.where(nn_probs >= 0.5, "vishing", "safe")

print("NN :", nn_labels)
print("NN probs:", np.round(nn_probs, 3))
